In [12]:
import pandas as pd
import numpy as np
import io
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

In [13]:
# 1. 파일 업로드
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

if file_name.endswith('.xlsx') or file_name.endswith('.xls'):
    df = pd.read_excel(io.BytesIO(uploaded[file_name]))
else:
    df = pd.read_csv(io.BytesIO(uploaded[file_name]), encoding='cp949')

df.columns = df.columns.astype(str).str.strip()

print("원본 데이터 모양:", df.shape)
print("컬럼명:", df.columns.tolist())

Saving 사고분석-서울.xlsx to 사고분석-서울 (1).xlsx
원본 데이터 모양: (1281, 22)
컬럼명: ['구분번호', '발생년월', '주야', '시군구', '사고내용', '사망자수', '중상자수', '경상자수', '부상신고자수', '사고유형', '법규위반', '노면상태', '기상상태', '도로형태', '가해운전자 차종', '가해운전자 성별', '가해운전자 연령대', '가해운전자 상해정도', '피해운전자 차종', '피해운전자 성별', '피해운전자 연령대', '피해운전자 상해정도']


In [14]:
# 2. 전처리

# 발생년월에서 year, month 추출
df['year'] = df['발생년월'].astype(str).str.extract(r'(\d{4})').astype(int)
df['month'] = df['발생년월'].astype(str).str.extract(r'(\d{1,2})월').astype(int)

# 시군구에서 구 단위 추출
df['District'] = df['시군구'].apply(
    lambda x: str(x).split()[1] if len(str(x).split()) > 1 else 'Unknown'
)

# 피해운전자 상해정도를 위험도 점수로 변환
injury_score_map = {
    '상해없음': 0,
    '부상신고': 33,
    '경상': 66,
    '중상': 100,
    '사망': 100,
    '기타불명': np.nan
}

df['injury_score'] = df['피해운전자 상해정도'].astype(str).map(injury_score_map)

# target이 없는 행 제거
df = df.dropna(subset=['injury_score']).copy()

print("\n--- 피해운전자 상해정도 분포 ---")
print(df['피해운전자 상해정도'].value_counts(dropna=False))

print("\n--- 위험도 점수 분포 ---")
print(df['injury_score'].value_counts().sort_index())


--- 피해운전자 상해정도 분포 ---
피해운전자 상해정도
경상      568
상해없음    336
중상      243
부상신고     75
사망        1
Name: count, dtype: int64

--- 위험도 점수 분포 ---
injury_score
0.0      336
33.0      75
66.0     568
100.0    244
Name: count, dtype: int64


In [15]:
# 3. Feature 구성
# 사고 결과를 직접 알려주는 컬럼은 제외한다.
# 사고내용, 사망자수, 중상자수, 경상자수, 부상신고자수, 피해운전자 상해정도 등은 target leakage 위험이 있다.

categorical_cols = [
    '주야',
    '기상상태',
    '노면상태',
    '도로형태',
    '사고유형',
    '법규위반',
    '가해운전자 차종',
    '가해운전자 성별',
    '가해운전자 연령대',
    'District'
]

categorical_cols = [col for col in categorical_cols if col in df.columns]

le_dict = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

features = ['year', 'month'] + categorical_cols

X = df[features]
y = df['injury_score']

print("\n사용한 feature:")
print(features)

print("\nX shape:", X.shape)
print("y shape:", y.shape)


사용한 feature:
['year', 'month', '주야', '기상상태', '노면상태', '도로형태', '사고유형', '법규위반', '가해운전자 차종', '가해운전자 성별', '가해운전자 연령대', 'District']

X shape: (1223, 12)
y shape: (1223,)


In [16]:
# 4. Train/Test split
# 희소한 상해정도 category 때문에 원문 기준 stratify는 에러가 날 수 있음
# 대신 injury_score 기준으로 큰 구간을 만들어 stratify

df['score_bin'] = pd.cut(
    df['injury_score'],
    bins=[-1, 24, 49, 74, 100],
    labels=['Low', 'Medium', 'High', 'Very High']
)

print(df['score_bin'].value_counts(dropna=False))

# 각 bin이 최소 2개 이상 있을 때만 stratify 적용
if df['score_bin'].value_counts().min() >= 2:
    stratify_option = df['score_bin']
else:
    stratify_option = None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=stratify_option,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

score_bin
High         568
Low          336
Very High    244
Medium        75
Name: count, dtype: int64
Train: (978, 12)
Test: (245, 12)


In [17]:
# 5. XGBoost Regressor 학습

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42
)

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [18]:
# 6. 예측 및 평가

y_pred = model.predict(X_test)

# 점수 범위를 0~100으로 제한
y_pred = np.clip(y_pred, 0, 100)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n--- XGBoost 위험도 점수 예측 결과 ---")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

result_df = pd.DataFrame({
    'actual_score': y_test.values,
    'predicted_score': y_pred
})

print("\n--- 실제 점수 vs 예측 점수 샘플 ---")
print(result_df.head(20))

print("\n--- 실제 점수별 예측 평균 ---")
print(result_df.groupby('actual_score')['predicted_score'].agg(['count', 'mean', 'std']))


--- XGBoost 위험도 점수 예측 결과 ---
MAE: 22.533007396483907
RMSE: 28.183331320084037
R2: 0.3950629579851007

--- 실제 점수 vs 예측 점수 샘플 ---
    actual_score  predicted_score
0           66.0        73.104935
1           66.0        70.757439
2           66.0        69.868279
3           33.0         8.460402
4           66.0        69.988068
5          100.0        76.223518
6            0.0        23.366041
7            0.0        30.658649
8           66.0        73.554779
9           66.0        71.010437
10          66.0        64.463081
11         100.0        72.889153
12          66.0        70.776070
13           0.0        34.465534
14          66.0        30.226900
15          33.0        68.958580
16          66.0        21.146639
17          33.0        75.908081
18           0.0        37.453178
19         100.0        74.709694

--- 실제 점수별 예측 평균 ---
              count       mean        std
actual_score                             
0.0              67  25.162247  12.628276
33.0     

In [19]:
# 7. 위험도 등급화
# 예측 점수를 Low / Medium / High / Very High로 나누기

def risk_grade(score):
    if score < 16.5:
        return '상해없음 수준'
    elif score < 49.5:
        return '부상신고 수준'
    elif score < 83:
        return '경상 수준'
    else:
        return '중상 수준'

result_df['predicted_grade'] = result_df['predicted_score'].apply(risk_grade)
result_df['actual_grade'] = result_df['actual_score'].apply(risk_grade)

print("\n--- 예측 위험도 등급 분포 ---")
print(result_df['predicted_grade'].value_counts())

print("\n--- 실제 위험도 등급 분포 ---")
print(result_df['actual_grade'].value_counts())


--- 예측 위험도 등급 분포 ---
predicted_grade
경상 수준      143
부상신고 수준     75
상해없음 수준     26
중상 수준        1
Name: count, dtype: int64

--- 실제 위험도 등급 분포 ---
actual_grade
경상 수준      114
상해없음 수준     67
중상 수준       49
부상신고 수준     15
Name: count, dtype: int64


In [20]:
# 8. Feature Importance

importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\n--- XGBoost Feature Importance ---")
print(importance_df)


--- XGBoost Feature Importance ---
      Feature  Importance
6        사고유형    0.489639
5        도로형태    0.097920
11   District    0.056425
2          주야    0.054804
7        법규위반    0.054103
0        year    0.048908
1       month    0.044757
9    가해운전자 성별    0.043707
10  가해운전자 연령대    0.041627
3        기상상태    0.038529
4        노면상태    0.029581
8    가해운전자 차종    0.000000


In [21]:
# 9. 예측 위험도 상위 사고 조건 확인

test_result = df.loc[X_test.index].copy()
test_result['actual_score'] = y_test.values
test_result['predicted_risk_score'] = y_pred

test_result = test_result.sort_values(
    by='predicted_risk_score',
    ascending=False
)

print("\n--- 예측 위험도 상위 20개 사고 조건 ---")
print(test_result[
    [
        'predicted_risk_score',
        'actual_score',
        'District',
        '주야',
        '기상상태',
        '노면상태',
        '도로형태',
        '사고유형',
        '법규위반',
        '가해운전자 성별',
        '가해운전자 연령대'
    ]
].head(20))


--- 예측 위험도 상위 20개 사고 조건 ---
      predicted_risk_score  actual_score  District  주야  기상상태  노면상태  도로형태  \
1206             84.502693         100.0         1   1     2     0     1   
1205             82.646210          66.0         1   0     2     0     3   
481              82.160637         100.0        24   1     2     0     1   
1106             81.842606          33.0         6   1     2     0     6   
960              80.244522         100.0        13   0     0     0     0   
480              79.399361          66.0        17   1     2     0     6   
1038             78.210304          66.0        14   1     2     1     2   
1082             77.824677          66.0         3   0     2     0     3   
704              77.773117          66.0         7   1     2     0     2   
485              77.719177          66.0        11   1     2     0     6   
203              77.493561         100.0        24   0     2     0     6   
470              77.414452          66.0        23   0     